# SCENIC MAF Analysis — 3 Focused Figures

1. **Fig 1**: MAF binarized regulon activity by `timepoint_condition` (Day1_MDI vs Day1_MDIX, ...)
2. **Fig 2**: MAF target connectivity network (motif-validated targets from `reg.csv`, Day 4)
3. **Fig 3**: LYVE1 importance under MAF across Day 1 → Day 4 trajectory



## 0. Imports & paths

In [ ]:
# SCENIC MAF-regulon figures; markdown-narrated twin of 08_scenic_regulons.
import scanpy as sc
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import loompy as lp
import json, zlib, base64, ast
import networkx as nx
from sklearn import mixture
from pathlib import Path


In [ ]:
SCENIC_LOOM = '/home/yanjun2/lyve1project/scanpy/8_figures/Module5_scenic/inte_scenic.loom'
ADATA_PATH  = '/home/yanjun2/lyve1project/scanpy/8_figures/inte_palantir.h5ad'
REG_DIR     = '/path/to/reg_files'   # CHANGE ME: folder with reg_day1.csv ... reg_day4.csv
OUT_DIR     = Path('/home/yanjun2/lyve1project/scanpy/8_figures/Module5_scenic/MAF_figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load adata

In [ ]:
adata = sc.read(ADATA_PATH)
print(adata)
print('Conditions before rename:', adata.obs['condition'].unique().tolist())


In [ ]:
adata.obs['condition'] = (adata.obs['condition'].astype(str)
                          .astype('category'))
print('Conditions after rename:', adata.obs['condition'].unique().tolist())


In [ ]:
adata.obs['timepoint_condition'] = (
    adata.obs['timepoint'].astype(str) + '_' + adata.obs['condition'].astype(str)
).astype('category')

# Order for plotting: Day1_MDI, Day1_MDIX, Day2_MDI, Day2_MDIX, ...
days = sorted(adata.obs['timepoint'].unique())
conds = ['MDI', 'MDIX']
order = [f'{d}_{c}' for d in days for c in conds
         if f'{d}_{c}' in adata.obs['timepoint_condition'].cat.categories]
adata.obs['timepoint_condition'] = adata.obs['timepoint_condition'].cat.reorder_categories(order)
print('timepoint_condition order:', order)


## 2. Load SCENIC loom → AUC matrix

In [ ]:
lf = lp.connect(SCENIC_LOOM, mode='r+', validate=False)
auc_mtx = pd.DataFrame(lf.ca.RegulonsAUC, index=lf.ca.CellID)
auc_mtx.columns = auc_mtx.columns.str.replace(r'\(', '_(', regex=True)
print('AUC matrix shape:', auc_mtx.shape)
print('First 5 regulons:', auc_mtx.columns[:5].tolist())


In [ ]:
# Sanity check: cell index must match adata
assert (auc_mtx.index == adata.obs_names).all(), "Cell index mismatch between AUC and adata!"
print('Cell index aligned ✓')


## 3. Binarize AUC matrix

Uses SCENIC author's GMM-based threshold derivation (Aibar et al. 2017), patched with `random_state=42` for reproducibility.

In [ ]:
def _derive_threshold(auc_mtx, regulon_name, random_state=42):
    data = auc_mtx[regulon_name].values.reshape(-1, 1)
    gmm = mixture.GaussianMixture(n_components=2, covariance_type='full',
                                   random_state=random_state).fit(data)
    avgs = gmm.means_
    stds = np.sqrt(gmm.covariances_.reshape(-1, 1))
    idx = np.argmax(avgs)
    threshold = max(avgs[idx] - 2 * stds[idx], 0)
    idx = np.argmin(avgs)
    lower_bound = avgs[idx] + 2 * stds[idx]
    return float(max(lower_bound, threshold))

def binarize(auc_mtx, threshold_overrides=None):
    thresholds = pd.Series({col: _derive_threshold(auc_mtx, col) for col in auc_mtx.columns})
    if threshold_overrides:
        thresholds.update(pd.Series(threshold_overrides))
    binarized = (auc_mtx > thresholds).astype(int)
    return binarized, thresholds


In [ ]:
binarized_auc, thresholds = binarize(auc_mtx)
print('Binarized matrix shape:', binarized_auc.shape)
print('MAF threshold:', thresholds.filter(like='MAF_(').to_dict())


## Figure 1 — MAF binarized activity by timepoint_condition

Two versions: heatmap (compact, annotated) + barplot (visually direct).

In [ ]:
# Locate MAF regulon column
maf_col = [c for c in binarized_auc.columns if c.startswith('MAF_(')][0]
print('MAF column:', maf_col)


In [ ]:
# Compute fraction of MAF+ cells per group
df = pd.DataFrame({
    'binarized': binarized_auc[maf_col].values,
    'group': adata.obs['timepoint_condition'].values,
})
maf_fraction = df.groupby('group', observed=True)['binarized'].mean().reindex(order)
print(maf_fraction)


In [ ]:
# Heatmap version
fig, ax = plt.subplots(figsize=(6, 1.2))
sns.heatmap(
    maf_fraction.to_frame().T,
    cmap='RdBu_r', center=maf_fraction.mean(),
    annot=True, fmt='.2f', cbar_kws={'label': 'Fraction MAF+'},
    ax=ax, linewidths=0.5, linecolor='white',
)
ax.set_yticklabels(['MAF regulon active'], rotation=0)
ax.set_xlabel('')
ax.set_title('MAF regulon activity (binarized)', fontsize=11, pad=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1a_MAF_binarized_heatmap.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Barplot version
fig, ax = plt.subplots(figsize=(5, 3))
colors = ['#888888' if g.split('_')[1] == 'MDI' else '#E65100' for g in maf_fraction.index]
ax.bar(maf_fraction.index, maf_fraction.values,
       color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Fraction MAF+ cells')
ax.set_ylim(0, 1)
ax.set_title('MAF regulon activity by timepoint × condition')
plt.xticks(rotation=45, ha='right')
sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1b_MAF_binarized_bar.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Figure 2 — MAF target connectivity network (Day 4, motif-validated)

Uses `reg_day4.csv` (motif-validated regulon from pySCENIC cisTarget step).
Highlights LYVE1+ resident and erythrophagocytic markers.

In [ ]:
def parse_regulon(reg_path, tf='MAF'):
    reg = pd.read_csv(reg_path, skiprows=3, header=None,
                      names=['TF', 'MotifID', 'AUC', 'NES', 'MotifSimQval',
                             'OrthologousIdentity', 'Annotation', 'Context',
                             'TargetGenes', 'RankAtMax'])
    targets = []
    for _, row in reg.query('TF == @tf').iterrows():
        try:
            targets.extend(ast.literal_eval(row['TargetGenes']))
        except (ValueError, SyntaxError):
            continue
    return (pd.DataFrame(targets, columns=['target', 'importance'])
            .groupby('target', as_index=False)['importance'].max()
            .sort_values('importance', ascending=False)
            .reset_index(drop=True))


In [ ]:
maf_targets_day4 = parse_regulon(f'{REG_DIR}/reg_day4.csv', tf='MAF')
print(maf_targets_day4.head(30))


In [ ]:
def plot_maf_network(targets_df, top_n=25, output='MAF_network.pdf',
                     highlight=None, tf='MAF', title_suffix=''):
    tf_top = targets_df.nlargest(top_n, 'importance').copy()
    imp = tf_top['importance']
    tf_top['width'] = 1 + 4 * (imp - imp.min()) / (imp.max() - imp.min())
    
    G = nx.DiGraph()
    G.add_node(tf)
    for _, r in tf_top.iterrows():
        G.add_edge(tf, r['target'], weight=r['width'])
    
    pos = nx.shell_layout(G, [[tf], [n for n in G.nodes if n != tf]])
    
    highlight = set(highlight or [])
    node_colors = ['#2E7D32' if n == tf
                   else '#E65100' if n in highlight
                   else '#CE93D8' for n in G.nodes]
    node_sizes = [2500 if n == tf else 600 for n in G.nodes]
    widths = [d['weight'] for _, _, d in G.edges(data=True)]
    
    fig, ax = plt.subplots(figsize=(11, 9))
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes,
                           edgecolors='black', linewidths=1.2, ax=ax)
    nx.draw_networkx_edges(
        G, pos, width=widths, edge_color='#555555', alpha=0.7,
        arrows=True, arrowsize=18, arrowstyle='-|>',
        connectionstyle='arc3,rad=0.05', node_size=node_sizes,
        min_source_margin=20, min_target_margin=20, ax=ax,
    )
    nx.draw_networkx_labels(
        G, pos, font_size=11,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1), ax=ax,
    )
    ax.set_title(f'{tf} regulon: top {top_n} motif-validated targets {title_suffix}',
                 fontsize=14, pad=20)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(output, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
plot_maf_network(
    maf_targets_day4, top_n=25,
    output=OUT_DIR / 'fig2_MAF_network_day4.pdf',
    highlight=['LYVE1', 'F13A1', 'MAFB', 'STAB1', 'CD163', 'CPT1B', 'MRC1', 'FOLR2'],
    title_suffix='(Day 4)',
)


## Figure 3 — LYVE1 importance trajectory under MAF (Day 1 → Day 4)

Tracks how LYVE1 (and other resident/ephago markers) is progressively recruited into the MAF regulon across differentiation.

In [ ]:
def get_target_importance(reg_path, tf, target):
    df = parse_regulon(reg_path, tf=tf)
    row = df.query('target == @target')
    return float(row['importance'].values[0]) if not row.empty else 0.0


In [ ]:
markers_to_track = ['LYVE1', 'F13A1', 'MRC1', 'FOLR2', 'STAB1', 'CD163', 'MAFB']

trajectory = pd.DataFrame(
    {gene: [get_target_importance(f'{REG_DIR}/reg_day{d}.csv', 'MAF', gene)
            for d in range(1, 5)]
     for gene in markers_to_track},
    index=[f'Day{d}' for d in range(1, 5)],
)
print(trajectory)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for gene in markers_to_track:
    is_lyve1 = gene == 'LYVE1'
    ax.plot(trajectory.index, trajectory[gene],
            marker='o', markersize=8 if is_lyve1 else 5,
            linewidth=2.5 if is_lyve1 else 1.2,
            color='#E65100' if is_lyve1 else None,
            alpha=1.0 if is_lyve1 else 0.6,
            label=gene, zorder=3 if is_lyve1 else 2)
ax.set_xlabel('Differentiation timepoint')
ax.set_ylabel('Target importance in MAF regulon')
ax.set_title('Recruitment of LYVE1+ resident markers into MAF regulon')
ax.legend(loc='upper left', frameon=False, fontsize=9)
ax.grid(axis='y', alpha=0.3)
sns.despine()
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig3_MAF_target_trajectory.pdf', dpi=300, bbox_inches='tight')
plt.show()


## Done

Files saved in `OUT_DIR`:
- `fig1a_MAF_binarized_heatmap.pdf`
- `fig1b_MAF_binarized_bar.pdf`
- `fig2_MAF_network_day4.pdf`
- `fig3_MAF_target_trajectory.pdf`